# Исследование случайного леса.
### Сравнительный анализ случайного леса, решающего дерева и линейной регрессии.

Датасет - Toyota Corolla. 

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, PolynomialFeatures
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, mean_absolute_error, r2_score
from sklearn.dummy import DummyRegressor

Загрузка данных

In [2]:
dataset = pd.read_csv('../hw04_linear_regression/ToyotaCorolla.csv')
dataset.head()

,Price,Age,KM,FuelType,HP,MetColor,Automatic,CC,Doors,Weight
0,13500,23,46986,Diesel,90,1,0,2000,3,1165
1,13750,23,72937,Diesel,90,1,0,2000,3,1165
2,13950,24,41711,Diesel,90,1,0,2000,3,1165
3,14950,26,48000,Diesel,90,0,0,2000,3,1165
4,13750,30,38500,Diesel,90,0,0,2000,3,1170


In [3]:
print("Количество записей:\n", dataset.count())
print("\nСтатистика:\n", dataset.describe())
print("\nПропуски:\n", dataset.isnull().sum())

Количество записей:
 Price        1436
Age          1436
KM           1436
FuelType     1436
HP           1436
MetColor     1436
Automatic    1436
CC           1436
Doors        1436
Weight       1436
dtype: int64

Статистика:
               Price          Age             KM           HP     MetColor  \
count   1436.000000  1436.000000    1436.000000  1436.000000  1436.000000   
mean   10730.824513    55.947075   68533.259749   101.502089     0.674791   
std     3626.964585    18.599988   37506.448872    14.981080     0.468616   
min     4350.000000     1.000000       1.000000    69.000000     0.000000   
25%     8450.000000    44.000000   43000.000000    90.000000     0.000000   
50%     9900.000000    61.000000   63389.500000   110.000000     1.000000   
75%    11950.000000    70.000000   87020.750000   110.000000     1.000000   
max    32500.000000    80.000000  243000.000000   192.000000     1.000000   

         Automatic           CC        Doors      Weight  
count  1436.000000 

Пропуски в данных отсутствуют.

EDA и корреляционный анализ приведён в ноутбуке с линейной регрессией. \
Перевод категориальноых признаков в bool:

In [4]:
dataset_encoded = pd.get_dummies(dataset, drop_first=True)

X = dataset_encoded.drop('Price', axis=1).values
y = dataset_encoded['Price'].values.reshape(-1,1)

Разделение выборки на train-test в соотношении 75%/25%.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print("Shape of X_train: ", X_train.shape)
print("Shape of X_test: ", X_test.shape)

Shape of X_train:  (1077, 10)
Shape of X_test:  (359, 10)


### Масштабирование данных: 
Необходимость масштабирования вызвана существенным отличием в порядках значений признаков(макс. разница  - 5 порядков). Масштабирование должно исключить риск роста значимости признака для предсказания, основываясь лишь на размере среднего значения по этому признаку.

In [6]:
ss = StandardScaler()
X_train_scaled = ss.fit_transform(X_train)
X_test_scaled = ss.transform(X_test)
y_train = np.array(y_train)
y_test = np.array(y_test)

print("X_train_scaled:\n", X_train_scaled)
print("X_test_scaled:\n", X_test_scaled)

X_train_scaled:
 [[ 1.07505623  3.28641953 -1.94110306 ...  1.55891304  2.78497712
  -2.61941098]
 [-0.90059013 -1.17636142  0.58429936 ...  0.14857303 -0.35906938
   0.38176522]
 [-2.50246556 -1.84784976  0.58429936 ...  0.04783445 -0.35906938
   0.38176522]
 ...
 [ 1.28863962  0.07433617  0.58429936 ... -0.05290412 -0.35906938
   0.38176522]
 [ 0.38091021 -0.11456263 -1.01069164 ... -1.06028984 -0.35906938
   0.38176522]
 [ 1.28863962  1.09948056  0.58429936 ... -0.75807413 -0.35906938
   0.38176522]]
X_test_scaled:
 [[-0.31323581 -1.24500805 -0.2796541  ... -0.95955127 -0.35906938
   0.38176522]
 [ 0.64788945  0.3070702   0.58429936 ... -0.35511984 -0.35906938
   0.38176522]
 [ 0.16732682  1.63536367 -1.94110306 ...  1.25669732  2.78497712
  -2.61941098]
 ...
 [ 1.28863962 -0.2274732   0.58429936 ...  0.83359532 -0.35906938
   0.38176522]
 [-1.11417352 -0.69572786  0.58429936 ... -0.45585841 -0.35906938
   0.38176522]
 [ 0.27411852  0.20723518 -1.01069164 ... -0.75807413 -0.35906938

Значения всех признаков теперь находятся в диапазоне [-1; 1].

In [7]:
def get_metrix(y_train, y_test, y_pred_train, y_pred_test):
    r2_score_train = r2_score(y_train, y_pred_train)
    r2_score_test = r2_score(y_test, y_pred_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    mape = mean_absolute_percentage_error(y_test, y_pred_test)
   
    print("Linear Regression Results:")
    print('R2_score (train): ', r2_score_train)
    print('R2_score (test): ', r2_score_test)
    print("RMSE: ", rmse)
    print("MAPE: ", mape)

    return {
        'r2_train' : r2_score_train,
        'r2_test': r2_score_test,
        'rmse' : rmse,
        'mape' : mape
    }

### Создание базовой модели случайного леса.

In [8]:
rfc = RandomForestRegressor()
rfc.fit(X_train_scaled, y_train)

rfc_pred_train = rfc.predict(X_train_scaled)
rfc_pred_test = rfc.predict(X_test_scaled)

rfc_metrics = get_metrix(y_train, y_test, rfc_pred_train, rfc_pred_test)

/home/ekattss/work/ML/ml_course_bsu/.venv/lib/python3.12/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Linear Regression Results:
R2_score (train):  0.9854698586118847
R2_score (test):  0.9147610924125062
RMSE:  1099.8253401723553
MAPE:  0.08724954404048729


Метрики модели показали более высокуюточность, чем метрики для линейной регрессии с различными модификациями.